# 04 — Trusting the judge: blind labels, kappa, and honest error bars

## The protocol (Block 4 is literally this)
1. A human (you) labels 40–60 calls on ONE binary question ("was the task completed?") **before ever seeing the judge's output on those calls** — *blind*, because seeing the judge first anchors you and the agreement number becomes circular.
2. Run the judge on the same calls. Align by `call_id`.
3. Compute agreement — but not raw agreement. **Kappa.**

## Why raw agreement lies
If 90% of calls succeeded, a broken judge that says "success" every single time scores 90% agreement while measuring nothing. **Cohen's kappa** asks: how much better than *chance* is the agreement, given each rater's base rates?

kappa = (p_observed − p_expected) / (1 − p_expected)

1.0 = perfect, 0 = exactly chance-level, negative = worse than chance. The Landis–Koch reading bands: 0.41–0.60 moderate, **0.61–0.80 substantial**, 0.81+ almost perfect. Our rule: claim "substantial" only if the number AND its confidence interval sit in the band. Otherwise say "moderate, directional" and keep your credibility.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
print("repo root:", ROOT)

import numpy as np
rng = np.random.default_rng(7)

def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    po = (a == b).mean()
    pe = (a.mean() * b.mean()) + ((1 - a.mean()) * (1 - b.mean()))
    return (po - pe) / (1 - pe)

n = 50
human = rng.integers(0, 2, n)
judge = np.where(rng.random(n) < 0.85, human, 1 - human)   # judge copies human, flips 15%
print(f"raw agreement: {(human == judge).mean():.2f}   kappa: {kappa(human, judge):.2f}")

**PREDICT before the next cell:** keep the SAME 85% copy-rate, but make the labels imbalanced (90% of calls succeed). Raw agreement stays ~0.85 — what happens to kappa, and why?

In [ ]:
human_skew = (rng.random(n) < 0.9).astype(int)
judge_skew = np.where(rng.random(n) < 0.85, human_skew, 1 - human_skew)
print(f"raw agreement: {(human_skew == judge_skew).mean():.2f}   kappa: {kappa(human_skew, judge_skew):.2f}")
print("\nsame raw agreement, weaker kappa - chance agreement is enormous when one class dominates.")
print("This is the prevalence problem. Knowing it = instant credibility with anyone who does evals.")

## Error bars by brute force: the bootstrap
50 items is small. A single kappa could be luck. The bootstrap: resample your 50 (with replacement) 1000 times, recompute kappa each time, take the 2.5th and 97.5th percentiles → a 95% CI without any distribution math. You have done this for benchmark variance; same tool, new metric.

In [ ]:
import matplotlib.pyplot as plt
ks = []
for _ in range(1000):
    idx = rng.integers(0, n, n)
    ks.append(kappa(human[idx], judge[idx]))
lo, hi = np.percentile(ks, [2.5, 97.5])
print(f"kappa = {kappa(human, judge):.2f}   95% CI [{lo:.2f}, {hi:.2f}]")
plt.figure(figsize=(7, 2.4)); plt.hist(ks, bins=40, color="#1D9E75")
plt.axvline(lo, color="#D85A30"); plt.axvline(hi, color="#D85A30")
plt.title("bootstrap distribution of kappa"); plt.show()
print("If 0.61 is inside your CI's lower half, you do NOT get to say 'substantial'. The CI decides, not hope.")

## The confusion matrix and the two disagreements
Agreement summarized is good; disagreement *itemized* is better. The 2×2 confusion matrix (human yes/no × judge yes/no) shows the error structure: does the judge miss failures (dangerous — false confidence) or invent them (annoying — alarm fatigue)? And the single most credibility-building demo move: **show two real cases where the judge was wrong and explain why.** It proves you tested the judge instead of worshipping it. The locked sentence: *"I'm not pretending this judge is magic; I tested where it agrees with humans and where it fails."*

In [ ]:
def confusion(a, b):
    m = np.zeros((2, 2), int)
    for x, y in zip(a, b):
        m[x, y] += 1
    return m

m = confusion(human, judge)
print("              judge=0  judge=1")
print(f"human=0       {m[0,0]:>5}  {m[0,1]:>5}")
print(f"human=1       {m[1,0]:>5}  {m[1,1]:>5}")
disagree = np.where(human != judge)[0]
print("\ndisagreement item indices:", disagree[:10], "<- in Block 7 these become the two slide cases")

## Self-check
1. Why must your labels be blind?
2. A judge agrees with you 88% raw on a 92%-positive dataset. Impressed? What do you compute?
3. Kappa 0.66, CI [0.48, 0.81] — what exactly are you allowed to claim?
4. Which confusion-matrix cell is most dangerous for a *failure-detection* product, and why?

<details><summary>Answers</summary>

1. Seeing the judge's answers first anchors your labels toward it; the agreement becomes self-fulfilling and worthless as evidence.
2. Not yet — chance agreement is ~85% at that prevalence. Compute kappa; it may be barely above zero.
3. "Moderate-to-substantial pilot agreement; the interval does not exclude moderate" — i.e., say "moderate, directional," never "substantial," because the CI dips well below 0.61.
4. human=1(failure present... depending on encoding) judged as fine — missed failures: the product's whole promise is catching them, and a leaky detector quietly restores false confidence.
</details>